In [0]:
def add(a,b):
    print (a+b)

In [0]:
num1 =10
num2 = int(input("Please enter the num2: "))

In [0]:
add(num1,num2)

In [0]:
Pyspark And DataBricks Que  

 

What is the difference between a narrow transformation and a wide transformation in PySpark, and how does each affect the execution plan of a Spark job? 

How does PySpark handle data partitioning across a cluster, and what strategies can be used to control the number of partitions in a DataFrame? 

What is the role of the Catalyst optimizer in PySpark, and how does it improve the performance of DataFrame and SQL-based operations? 

How does PySpark's lazy evaluation model work, and what are the practical implications of this model when building data pipelines? 

What are the key differences between using RDDs and DataFrames in PySpark, and in what scenarios would you prefer one over the other? 

How would you handle skewed data in a PySpark join operation, and what techniques can be applied to avoid performance bottlenecks caused by data skew? 

What is a broadcast join in PySpark, and under what conditions would you choose to use it over a standard shuffle join? 

How does PySpark's structured streaming differ from batch processing, and what are the key components involved in building a streaming pipeline using PySpark? 

What are accumulators and broadcast variables in PySpark, and how are they used to optimize distributed computations? 

How does caching and persistence work in PySpark, and what storage levels are available when persisting a DataFrame or RDD? 

## 1. Narrow vs Wide Transformations

**Narrow Transformation**: Each input partition contributes to only one output partition. No data shuffling across nodes.
- Examples: `map()`, `filter()`, `select()`, `withColumn()`

**Wide Transformation**: Each input partition contributes to multiple output partitions. Requires shuffling data across the cluster.
- Examples: `groupBy()`, `join()`, `repartition()`, `orderBy()`

**Scenario**: Processing 1 billion customer transactions
```python
# Narrow - filters data in place, no shuffle
df.filter(col("amount") > 100)  # Fast, processes each partition independently

# Wide - requires shuffle across all nodes
df.groupBy("customer_id").agg(sum("amount"))  # Slower, data moves across network
```

**Execution Impact**: Wide transformations create stage boundaries in the execution plan, causing network I/O and potential bottlenecks.

---

## 2. Data Partitioning in PySpark

**How it works**: Spark distributes data across cluster nodes in partitions. Each partition is processed by a single core.

**Default partitioning**: 
- Reading files: 1 partition per file (or split for large files)
- Post-shuffle: `spark.sql.shuffle.partitions` (default 200)

**Control strategies**:
```python
# 1. Set partition count during read
df = spark.read.parquet("s3://bucket/data").repartition(100)

# 2. Control shuffle partitions globally
spark.conf.set("spark.sql.shuffle.partitions", "50")

# 3. Partition by specific column (for joins/aggregations)
df.repartition(10, "customer_id")  # Co-locates same customer_id

# 4. Reduce partitions without shuffle
df.coalesce(5)  # Combines partitions, faster than repartition
```

**Scenario**: Processing 100GB dataset on 10-node cluster with 4 cores each = 40 cores
- Too few partitions (10): Underutilizes cluster, only 10 cores active
- Too many partitions (10,000): Too much overhead scheduling tiny tasks
- Optimal: 80-120 partitions (2-3x number of cores)

---

## 3. Catalyst Optimizer Role

**What it is**: Query optimization engine that converts DataFrame/SQL operations into an efficient physical execution plan.

**Optimization phases**:
1. **Analysis**: Resolves columns, tables, functions
2. **Logical optimization**: Predicate pushdown, constant folding, column pruning
3. **Physical planning**: Chooses join strategies, broadcast decisions
4. **Code generation**: Compiles to optimized JVM bytecode

**Scenario**: E-commerce order analysis
```python
# Your code
df = spark.table("orders").filter(col("year") == 2024) \
    .join(spark.table("customers"), "customer_id") \
    .select("order_id", "customer_name", "total")

# Catalyst optimizations:
# 1. Predicate pushdown: Filters year=2024 before join (reads less data)
# 2. Column pruning: Only reads needed columns, not entire tables
# 3. Join reordering: May join smaller filtered table first
# 4. Broadcast join: If customers table < 10MB, broadcasts instead of shuffle
```

**Performance gain**: 10x-100x faster by reducing data scanned and moved.

---

## 4. Lazy Evaluation Model

**How it works**: Transformations are NOT executed immediately. Spark builds a DAG (Directed Acyclic Graph) of operations and only executes when an action is called.

**Transformations** (lazy): `select()`, `filter()`, `groupBy()`, `join()`
**Actions** (trigger execution): `show()`, `count()`, `collect()`, `write()`

**Scenario**: Data quality pipeline
```python
# Step 1-4: No execution yet (lazy)
df1 = spark.read.csv("raw_data.csv")           # 1. Define read
df2 = df1.filter(col("status") == "active")     # 2. Plan filter
df3 = df2.withColumn("date", current_date())    # 3. Plan column add
df4 = df3.select("id", "name", "date")          # 4. Plan projection

# Only when action called, entire DAG executes optimally
df4.count()  # NOW execution happens, Catalyst optimizes entire pipeline
```

**Practical implications**:
- **Pro**: Catalyst can optimize entire pipeline, not individual steps
- **Pro**: Avoids computing unused intermediate results
- **Con**: Errors appear at action time, not where written
- **Con**: Debugging requires inserting actions like `df.cache()` or `df.count()`

---

## 5. RDDs vs DataFrames

| Aspect | RDD | DataFrame |
|--------|-----|----------|
| **API** | Low-level, functional | High-level, SQL-like |
| **Schema** | No schema | Structured schema |
| **Optimization** | Manual | Catalyst optimizer |
| **Performance** | Slower | 2-10x faster |
| **Type safety** | Compile-time (Scala) | Runtime |
| **Serialization** | Java serialization | Tungsten binary format |

**When to use RDD**:
1. Need fine-grained control over data (complex algorithms)
2. Working with unstructured data that doesn't fit tabular model
3. Legacy code from older Spark versions

**When to use DataFrame**:
1. Structured/semi-structured data (CSV, JSON, Parquet)
2. SQL-style operations (joins, aggregations, filters)
3. Need automatic optimization
4. Working with multiple languages (interoperable)

**Scenario**: Text processing vs structured analytics
```python
# Use RDD: Custom text processing algorithm
rdd = sc.textFile("logs.txt") \
    .map(lambda line: complex_parse(line)) \
    .filter(lambda x: custom_logic(x))

# Use DataFrame: Analytics on structured data
df = spark.read.parquet("sales.parquet") \
    .groupBy("region").agg(sum("revenue")) \
    .orderBy(desc("sum(revenue)"))
```

---

## 6. Handling Skewed Data in Joins

**Problem**: When one key has disproportionately many records, causing one executor to process most data while others idle.

**Scenario**: Joining orders to customers, where customer_id "UNKNOWN" has 80% of orders.

**Techniques**:

**1. Salting** (add random suffix to distribute hot keys)
```python
from pyspark.sql.functions import rand, concat, lit

# Add salt to skewed key
orders_salted = orders.withColumn("salted_id", 
    concat(col("customer_id"), lit("_"), (rand() * 10).cast("int")))

# Replicate smaller table
customers_replicated = customers.withColumn("salt", explode(array(*[lit(i) for i in range(10)]))) \
    .withColumn("salted_id", concat(col("customer_id"), lit("_"), col("salt")))

result = orders_salted.join(customers_replicated, "salted_id")
```

**2. Broadcast join** (if skewed table is small)
```python
from pyspark.sql.functions import broadcast
df = large_df.join(broadcast(small_skewed_df), "key")
```

**3. Separate skewed keys**
```python
# Process normal and skewed keys separately
normal_orders = orders.filter(col("customer_id") != "UNKNOWN")
skewed_orders = orders.filter(col("customer_id") == "UNKNOWN")

normal_result = normal_orders.join(customers, "customer_id")
skewed_result = skewed_orders.crossJoin(customers.filter(col("customer_id") == "UNKNOWN"))

final = normal_result.union(skewed_result)
```

**4. Adaptive Query Execution (AQE)** - Spark 3.0+
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# Spark automatically detects and handles skew
```

---

## 7. Broadcast Join

**What it is**: Sends a copy of the smaller table to every executor, avoiding shuffle of the larger table.

**When to use**:
- One table fits in memory (< spark.sql.autoBroadcastJoinThreshold, default 10MB)
- Joining large table with dimension/lookup table
- Need to avoid expensive shuffle

**Scenario**: 1TB fact table joining with 5MB dimension table
```python
from pyspark.sql.functions import broadcast

# BAD: Standard shuffle join - moves 1TB across network
result = huge_sales.join(small_products, "product_id")

# GOOD: Broadcast join - sends 5MB to all executors
result = huge_sales.join(broadcast(small_products), "product_id")

# Automatic broadcast (if < 10MB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)  # 10MB
```

**Performance impact**: 
- Shuffle join: Minutes to hours (depends on data size)
- Broadcast join: Seconds (no shuffle, just broadcast 5MB)

**Limitations**: 
- Broadcast table must fit in executor memory
- Not suitable for large-to-large joins

---

## 8. Structured Streaming vs Batch Processing

**Key Differences**:

| Aspect | Batch | Streaming |
|--------|-------|----------|
| **Data** | Bounded (historical) | Unbounded (continuous) |
| **Execution** | One-time job | Continuous query |
| **Latency** | Minutes to hours | Sub-second to minutes |
| **API** | `spark.read` | `spark.readStream` |
| **Output** | `df.write` | `df.writeStream` |

**Structured Streaming Components**:

```python
# 1. Source: Where data comes from
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "transactions") \
    .load()

# 2. Transformation: Same as batch
processed = df.select("value") \
    .withColumn("timestamp", current_timestamp()) \
    .groupBy(window("timestamp", "5 minutes")) \
    .agg(count("*").alias("count"))

# 3. Sink: Where data goes
query = processed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoint") \
    .trigger(processingTime="10 seconds") \
    .start("/tmp/output")

# 4. Trigger: When to process
# - processingTime: micro-batch every N seconds
# - once: single batch then stop
# - continuous: experimental low-latency mode

# 5. Checkpoint: Fault tolerance
# Stores offset and state to recover from failures
```

**Scenario**: Real-time fraud detection
- Batch: Analyze yesterday's transactions (too late!)
- Streaming: Detect fraud within seconds of transaction

---

## 9. Accumulators and Broadcast Variables

**Accumulators**: Shared variables that executors can ADD to (only driver can read)
```python
# Count invalid records across all executors
invalid_count = spark.sparkContext.accumulator(0)

def process_row(row):
    if not is_valid(row):
        invalid_count.add(1)
    return row

df.foreach(process_row)
print(f"Invalid records: {invalid_count.value}")  # Read on driver
```

**Broadcast Variables**: Read-only shared data sent to all executors once
```python
# Load 100MB lookup table once, not per task
lookup_dict = {"key1": "value1", "key2": "value2", ...}  # 100MB
broadcast_lookup = spark.sparkContext.broadcast(lookup_dict)

def enrich_row(row):
    # Access broadcast variable efficiently
    enriched_value = broadcast_lookup.value.get(row.key)
    return enriched_value

df.rdd.map(enrich_row)
```

**Scenario**: Processing 1 billion records with 100MB lookup table
- Without broadcast: Sends 100MB × 1000 tasks = 100GB network transfer
- With broadcast: Sends 100MB × 10 executors = 1GB network transfer (100x less!)

---

## 10. Caching and Persistence

**Purpose**: Store intermediate results in memory/disk to avoid recomputing in iterative algorithms.

**Storage Levels**:
```python
from pyspark import StorageLevel

# 1. MEMORY_ONLY (default for cache())
df.cache()  # Same as df.persist(StorageLevel.MEMORY_ONLY)
# Fast but can lose data if memory full

# 2. MEMORY_AND_DISK
df.persist(StorageLevel.MEMORY_AND_DISK)
# Spills to disk if memory insufficient

# 3. MEMORY_ONLY_SER (serialized)
df.persist(StorageLevel.MEMORY_ONLY_SER)
# More space-efficient, slower access

# 4. DISK_ONLY
df.persist(StorageLevel.DISK_ONLY)
# For very large datasets

# 5. OFF_HEAP
df.persist(StorageLevel.OFF_HEAP)
# Uses off-heap memory (Tungsten)
```

**When to use**:
```python
# Machine learning - iterate same data multiple times
training_data = spark.read.parquet("features.parquet").cache()

for i in range(10):
    model.fit(training_data)  # Reads from cache, not disk

# Complex pipeline - reuse expensive computation
expensive_df = raw_df.join(dim1).join(dim2).filter(...).cache()

result1 = expensive_df.groupBy("col1").count()
result2 = expensive_df.groupBy("col2").sum("amount")
# Both reuse cached expensive_df

# Don't forget to unpersist when done
expensive_df.unpersist()
```

**Best practices**:
- Cache after expensive operations (joins, aggregations)
- Cache before multiple actions on same data
- Don't cache everything (memory waste)
- Monitor cache usage in Spark UI Storage tab

**Scenario**: Iterative K-means clustering
- Without cache: Reads 100GB from disk 10 times = 1TB I/O
- With cache: Reads 100GB once, then memory access = 100GB I/O (10x faster)